# ✨ LivePortrait Studio - Lái Chuyển Động Cho Ảnh Nhân Vật (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Thời gian chạy (Runtime)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm nút ▶️ ở ô lệnh bên dưới -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ tải ~2GB trọng số và lưu vào Drive, **từ lần thứ 2 trở đi sẽ nạp tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title Chạy LivePortrait WebUI 1-Click (Tự Động Caching Google Drive)
import os
import shutil
from google.colab import drive
from IPython.display import clear_output

# 1. Gắn kết Google Drive thông minh
print("🔗 Đang kiểm tra kết nối Google Drive...")
if not os.path.exists('/content/drive/MyDrive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Không thể kết nối Drive tự động ({e}). Sẽ chạy tạm thời trên máy ảo.")

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/LivePortrait"
drive_weights_dir = f"{drive_cache_dir}/pretrained_weights"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)

%cd /content
if not os.path.exists("/content/LivePortrait"):
    !git clone -b dev https://github.com/camenduru/LivePortrait /content/LivePortrait

%cd /content/LivePortrait

# 2. Kiểm tra xem 2GB weights đã có trên Drive chưa
if os.path.exists(drive_weights_dir) and os.path.isdir(drive_weights_dir) and len(os.listdir(drive_weights_dir)) > 3:
    print("🎉 ĐÃ TÌM THẤY 2GB TRỌNG SỐ TRONG GOOGLE DRIVE! Nạp trực tiếp không cần tải lại...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !cp -r "{drive_weights_dir}" /content/LivePortrait/pretrained_weights
else:
    print("⏳ Đang tải trọng số AI lần đầu (~2GB) và lưu vào Google Drive của bạn...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !git clone https://huggingface.co/camenduru/LivePortrait /content/LivePortrait/pretrained_weights
    print("💾 Đang lưu bản sao trọng số vào Google Drive để lần sau nạp ngay lập tức...")
    os.makedirs(drive_cache_dir, exist_ok=True)
    !cp -r /content/LivePortrait/pretrained_weights "{drive_cache_dir}/"

# 3. Cài đặt thư viện môi trường
print("📦 Đang chuẩn bị môi trường...")
!pip install -q tyro onnxruntime-gpu onnx gradio colorama ffmpeg-python

# 4. Biên dịch module Cython 3D
%cd /content/LivePortrait/src/utils/dependencies/insightface/thirdparty/face3d/mesh/cython
!python setup.py build_ext --inplace

%cd /content/LivePortrait

# 5. Vá lỗi PyTorch 2.6 (Bắt buộc weights_only=False khi nạp trọng số cũ)
!sed -i "s/torch.load(ckpt_path/torch.load(ckpt_path, weights_only=False/g" /content/LivePortrait/src/utils/helper.py
!python -c "content = open('/content/LivePortrait/app.py').read(); open('/content/LivePortrait/app.py', 'w').write('import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **dict(k, weights_only=False))\n' + content) if '_old_load' not in content else None"

clear_output()
print("🚀 Đang khởi động LivePortrait WebUI...")
!python app.py --share
